In [5]:
import pygame
import math
import random
import sys

# Настройки экрана
WIDTH, HEIGHT = 1200, 800
FPS = 60

# Цвета
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
GREEN = (0, 255, 0)
BLUE = (0, 0, 255)

# Генетические настройки
POPULATION_SIZE = 15
MUTATION_RATE = 0.1

class NeuralNetwork:
    """Простая нейросеть: 5 входов (радары), 4 скрытых узла, 2 выхода (влево/вправо)"""
    def __init__(self, weights1=None, weights2=None):
        if weights1 is None:
            self.w1 = [[random.uniform(-1, 1) for _ in range(4)] for _ in range(5)]
            self.w2 = [[random.uniform(-1, 1) for _ in range(2)] for _ in range(4)]
        else:
            self.w1 = weights1
            self.w2 = weights2

    def forward(self, inputs):
        # Слой 1 (Скрытый)
        hidden = [0.0] * 4
        for j in range(4):
            s = sum(inputs[i] * self.w1[i][j] for i in range(5))
            hidden[j] = math.tanh(s)
        
        # Слой 2 (Выходной)
        outputs = [0.0] * 2
        for j in range(2):
            s = sum(hidden[i] * self.w2[i][j] for i in range(4))
            outputs[j] = math.tanh(s)
        
        return outputs

    def mutate(self):
        """Случайное изменение весов (мутация)"""
        for i in range(5):
            for j in range(4):
                if random.random() < MUTATION_RATE:
                    self.w1[i][j] += random.uniform(-0.5, 0.5)
        for i in range(4):
            for j in range(2):
                if random.random() < MUTATION_RATE:
                    self.w2[i][j] += random.uniform(-0.5, 0.5)

class Car:
    def __init__(self, brain=None):
        # Стартовая позиция чуть смещена, чтобы машинка сразу не задевала стену
        self.x = 150.0
        self.y = 250.0
        self.angle = math.pi / 2  # Направляем машинку изначально вниз по треку
        self.speed = 4.0
        self.alive = True
        self.distance_traveled = 0.0
        self.time_alive = 0
        
        self.radars = [0.0] * 5
        self.brain = brain if brain else NeuralNetwork()

    def update(self, track_surface):
        if not self.alive:
            return

        self.time_alive += 1
        self.distance_traveled += self.speed

        self.check_radars(track_surface)
        
        inputs = [r / 200.0 for r in self.radars]
        outputs = self.brain.forward(inputs)
        
        # ИСПРАВЛЕНО: Проверяем правильные индексы outputs[0] и outputs[1]
        if outputs[0] > 0.5:
            self.angle -= 0.05  # Поворот влево
        elif outputs[1] > 0.5:
            self.angle += 0.05  # Поворот вправо

        # Движение вперед
        self.x += math.cos(self.angle) * self.speed
        self.y += math.sin(self.angle) * self.speed

        self.check_collision(track_surface)
        
        # Защита от застревания (если крутится на месте)
        if self.time_alive > 300 and self.distance_traveled < 50:
            self.alive = False

    def check_radars(self, surface):
        radar_angles = [-0.6, -0.3, 0.0, 0.3, 0.6]
        
        for i, angle_offset in enumerate(radar_angles):
            length = 0
            while length < 200:
                length += 2
                target_x = int(self.x + math.cos(self.angle + angle_offset) * length)
                target_y = int(self.y + math.sin(self.angle + angle_offset) * length)
                
                if target_x < 0 or target_x >= WIDTH or target_y < 0 or target_y >= HEIGHT:
                    break
                if surface.get_at((target_x, target_y)) == WHITE:
                    break
            self.radars[i] = length

    def check_collision(self, surface):
        # Проверяем передний и задний бампер машинки
        points = [
            (int(self.x + math.cos(self.angle)*10), int(self.y + math.sin(self.angle)*10)),
            (int(self.x - math.cos(self.angle)*10), int(self.y - math.sin(self.angle)*10))
        ]
        for p in points:
            if p[0] < 0 or p[0] >= WIDTH or p[1] < 0 or p[1] >= HEIGHT:
                self.alive = False
                return
            if surface.get_at(p) == WHITE:
                self.alive = False
                return

    def draw(self, screen):
        if not self.alive:
            return
        pygame.draw.circle(screen, GREEN, (int(self.x), int(self.y)), 8)
        
        front_x = int(self.x + math.cos(self.angle) * 12)
        front_y = int(self.y + math.sin(self.angle) * 12)
        pygame.draw.line(screen, RED, (int(self.x), int(self.y)), (front_x, front_y), 2)


def draw_track(surface):
    surface.fill(WHITE)
    pygame.draw.ellipse(surface, BLACK, (50, 50, WIDTH - 100, HEIGHT - 100))
    pygame.draw.ellipse(surface, WHITE, (200, 200, WIDTH - 400, HEIGHT - 400))


def evolve_population(cars):
    # Сортируем по успешности (кто дальше уехал)
    cars.sort(key=lambda c: c.distance_traveled, reverse=True)
    
    # Отбираем двух лучших родителей
    best_parent_1 = cars[0].brain
    best_parent_2 = cars[1].brain if len(cars) > 1 else cars[0].brain
    
    new_cars = []
    for i in range(POPULATION_SIZE):
        if i < 2:
            # Сохраняем двух лучших без изменений (элитарность)
            new_brain = NeuralNetwork(weights1=cars[i].brain.w1, weights2=cars[i].brain.w2)
        else:
            # Скрещивание весов
            child_w1 = []
            for r in range(5):
                row = []
                for c in range(4):
                    row.append(best_parent_1.w1[r][c] if random.random() > 0.5 else best_parent_2.w1[r][c])
                child_w1.append(row)
                
            child_w2 = []
            for r in range(4):
                row = []
                for c in range(2):
                    row.append(best_parent_1.w2[r][c] if random.random() > 0.5 else best_parent_2.w2[r][c])
                child_w2.append(row)
                
            new_brain = NeuralNetwork(weights1=child_w1, weights2=child_w2)
            new_brain.mutate()
            
        new_cars.append(Car(brain=new_brain))
        
    return new_cars


def main():
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("AI Гоночные Машинки (Генетический алгоритм)")
    clock = pygame.time.Clock()
    font = pygame.font.SysFont("Arial", 24)

    track_surface = pygame.Surface((WIDTH, HEIGHT))
    draw_track(track_surface)

    cars = [Car() for _ in range(POPULATION_SIZE)]
    generation = 1

    run = True
    while run:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run = False
                pygame.quit()
                sys.exit()

        any_alive = False
        for car in cars:
            car.update(track_surface)
            if car.alive:
                any_alive = True

        if not any_alive:
            cars = evolve_population(cars)
            generation += 1

        screen.blit(track_surface, (0, 0))
        
        for car in cars:
            car.draw(screen)

        text = font.render(f"Поколение: {generation}", True, BLUE)
        screen.blit(text, (30, 30))

        pygame.display.flip()

if __name__ == "__main__":
    main()


SystemExit: 